In [1]:
import pandas as pd
from pyopenms_viz.util import download_file
from matchms.importing import load_from_msp
from plotly.subplots import make_subplots
from plotly.offline import iplot
import pyopenms as oms
import numpy as np
import pyarrow.parquet as pq

pd.options.plotting.backend = "ms_plotly"


In [ ]:
peak_table1 = pd.read_parquet("peaks-tables/29_qc_no_dil_milliq.parquet")
peak_table2 = pd.read_parquet("peaks-tables/27_qc_2x_dil_milliq.parquet")
peak_table3 = pd.read_parquet("peaks-tables/28_qc_64x_dil_milliq.parquet")

In [3]:
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[],
    shared_xaxes='all',
    shared_yaxes='all'
)

In [ ]:
peakmap1 = peak_table1.plot(kind="peakmap", x="rt", y="mz", z="area")
peakmap2 = peak_table2.plot(kind="peakmap", x="rt", y="mz", z="area")
peakmap3 = peak_table3.plot(kind="peakmap", x="rt", y="mz", z="area")

In [ ]:
fig.append_trace(peakmap1['data'][0], 1, 1)
fig.append_trace(peakmap2['data'][0], 1, 2)
fig.append_trace(peakmap3['data'][0], 2, 1)

In [6]:
fig

In [2]:
lib = list(load_from_msp("rcx_gc-orbitrap_metabolites_20210817.msp"))
acetylserotonin_3TMS = lib[2]

In [9]:
peaks_df = pd.DataFrame(acetylserotonin_3TMS.peaks.to_numpy.astype(np.float32), columns = ["mz", "intensity"])
peaks_df.head()

,mz,intensity
0,73.046783,1695472.0
1,75.026070,264489.0
2,102.073334,352935.0
3,147.065491,217538.0
4,149.044754,247633.0


In [11]:
exp = oms.MSExperiment()
oms.MzMLFile().load("8_qc_no_dil_milliq.mzml", exp)


df = exp.get_df(long=True).sort_values(by="mz")  # default: long = False
df.head(2)

,RT,mz,inty,ms_level
4521091,205.532471,69.303795,0.0,1
1157349,145.583725,69.303810,0.0,1


In [12]:
eics = pd.merge_asof(df, peaks_df, left_on = "mz", right_on="mz", tolerance=0.003)

In [13]:
eics.head()

,RT,mz,inty,ms_level,intensity
0,205.532471,69.303795,0.0,1,NaN
1,145.583725,69.303810,0.0,1,NaN
2,255.525757,69.303848,0.0,1,NaN
3,120.796776,69.303879,0.0,1,NaN
4,173.969803,69.303894,0.0,1,NaN


In [ ]:
eics.plot(
    kind="chromatogram",
    x="RT",
    y="inty",
    by="mz",
    legend_config=dict(bbox_to_anchor=(1, 0.7)),
)

KeyError: 'Column `rt` not in data, `x` could not be set'